# Predictive Process Monitoring — Suffix Prediction Benchmark

This notebook provides a **single entry point** to reproduce all experiments from the paper:

> *SuTraN: an Encoder-Decoder Transformer for Full-Context-Aware Suffix Prediction of Business Processes*  
> ICPM 2024

It also includes the **BEST** baseline introduced in:

> *BEST: Bilaterally Expanding Subtrace Tree for Event Sequence Prediction*  
> Rauch et al., BPM 2025

---

## What this notebook does

1. **Creates pre-processed tensor datasets** from the raw event logs (BPIC17, BPIC17-DR, BPIC19).
2. **Trains and evaluates every model** on each event log and saves results to disk.

Each model section includes a short description, the paper it is based on, and a code cell that runs `train_eval()` for every log.

---

## Models covered

| # | Model | Type | Paper |
|---|-------|------|-------|
| 1 | **SuTraN (DA)** | Encoder-Decoder Transformer | SuTraN paper (ICPM 2024) |
| 2 | **SuTraN (NDA)** | Encoder-Decoder Transformer (control-flow only) | SuTraN paper (ICPM 2024) |
| 3 | **CRTP-LSTM (DA)** | Multi-task LSTM | Camargo et al. (2019) |
| 4 | **CRTP-LSTM (NDA)** | Multi-task LSTM (control-flow only) | Camargo et al. (2019) |
| 5 | **ED-LSTM** | Encoder-Decoder LSTM (seq2seq) | Tax et al. (2017) / Pfeiffer et al. (2021) |
| 6 | **SEP-LSTM** | One-step-ahead LSTM with iterative feedback | Taymouri et al. (2021) |
| 7 | **BEST** | N-gram pattern tree (no neural network) | Rauch et al., BPM 2025 |

**DA** = Data-Aware (uses all event / case attributes)  
**NDA** = Non-Data-Aware (uses only activity labels and timestamp proxies)

Data leakage prevention:
1. Weytjens: Creating Unbiased Public Benchmark Datasets with Data Leakage Prevention for Predictive Process Monitoring.

---


---
# Part 1 — Creating Datasets

Each script reads a raw CSV event log, applies log-specific preprocessing (timestamp parsing, invalid-case removal, feature engineering), then calls `log_to_tensors()` from `Preprocessing/from_log_to_tensors.py`.

The pipeline:
1. Filters cases by date range and maximum duration.
2. Maps categorical features to integers.
3. Standardises numerical features using training-set statistics.
4. Creates all prefix–suffix pairs (sliding window).
5. Splits into train / validation / test using an **out-of-time** split with data-leakage prevention.
6. Saves `train_tensordataset.pt`, `val_tensordataset.pt`, `test_tensordataset.pt` and metadata pickles inside a subfolder named after the log.

> **Note:** Run each cell only once. Re-running will overwrite the saved tensors.

# Any dataset

In [21]:
from create_general_data import construct_datasets

# ------------------------------------------------------------------ #
# Edit the variables below to match your event log.                   #
# ------------------------------------------------------------------ #

LOG_PATH   = 'Logs/Electronic Invoicing.xes'   # or 'my_log.csv'
LOG_NAME   = 'Electronic Invoicing'

# Standard XES column names — change if your CSV uses different names.
CASE_ID    = 'case:concept:name'
ACT_LABEL  = 'concept:name'
TIMESTAMP  = 'time:timestamp'

# Set to None to auto-detect from column prefixes / dtypes, or provide
# explicit lists as in the log-specific create_*.py files.
CAT_CASEFTS  = None
NUM_CASEFTS  = None
CAT_EVENTFTS = None
NUM_EVENTFTS = None

# Filtering / splitting parameters — adjust to your log's date range
# and case-length distribution.
START_DATE        = None   # e.g. "2018-01"
START_BEFORE_DATE = None   # e.g. "2018-09"
END_DATE          = None   # e.g. "2019-02"
MAX_DAYS          = None   # e.g. 143.33
WINDOW_SIZE       = None   # None → auto (98.5th percentile)
TEST_LEN_SHARE    = 0.20
VAL_LEN_SHARE     = 0.20
MODE              = 'preferred' # preferred or workaround
OUTCOME           = None
PLOT              = True    # set to False to skip the split visualisation

construct_datasets(
    log_path=LOG_PATH,
    log_name=LOG_NAME,
    case_id=CASE_ID,
    act_label=ACT_LABEL,
    timestamp=TIMESTAMP,
    cat_casefts=CAT_CASEFTS,
    num_casefts=NUM_CASEFTS,
    cat_eventfts=CAT_EVENTFTS,
    num_eventfts=NUM_EVENTFTS,
    outcome=OUTCOME,
    start_date=START_DATE,
    start_before_date=START_BEFORE_DATE,
    end_date=END_DATE,
    max_days=MAX_DAYS,
    window_size=WINDOW_SIZE,
    test_len_share=TEST_LEN_SHARE,
    val_len_share=VAL_LEN_SHARE,
    mode=MODE,
    plot=PLOT,
)

parsing log, completed traces ::   0%|          | 0/20135 [00:00<?, ?it/s]

Split plot saved to 'results_per_log/Electronic Invoicing/Electronic Invoicing_preferred_split.png'
Auto-detected features:
  cat_casefts  : ['case:description']
  num_casefts  : []
  cat_eventfts : ['org:resource', 'lifecycle:transition']
  num_eventfts : []
Auto-derived window_size (98.5th percentile): 20
Auto-derived max_days (maximum case duration): 2.65
Generating Dataframes...


100%|█████████████████████████████████████████████████| 4/4 [00:00<00:00, 395.68it/s]


New-block labels saved to 'results_per_log/Electronic Invoicing/'.
Cases – train: 12873  val: 3219  test: 4043
Generating Tensors...
Computing train set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
____________________________
Computing validation set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
____________________________
Computing test set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
Pairs – train: 98806  val: 24743  test: 30883
Tensors saved to 'results_per_log/Electronic Invoicing/'
tss_index = 0  (= 0 num_casefts + 0 num_eventfts)
Use this value for the tss_index parameter in TRAIN_EVAL_*.py scripts.
Concurrent-trace test samples: 30883 / 30883


/workspace/baselines/SuffixTransformerNetwork/create_general_data.py:532: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  _nb = torch.load(os.path.join(output_directory, 'test

{'n_train': 12873,
 'train_pairs': 98806,
 'n_val': 3219,
 'val_pairs': 24743,
 'n_test': 4043,
 'test_pairs': 30883,
 'conc_count': 30883}

# Preproses all datasets sutran

In [1]:
from create_general_data import run_all_logs

folder = "Logs/"
output_file = "split_summary_sutran_best.csv"
run_all_logs(folder, output_file)

Processing 8 logs with 8 workers ...
Auto-detected features:
  cat_casefts  : []
  num_casefts  : []
  cat_eventfts : ['InfectionSuspected', 'org:group', 'DiagnosticBlood', 'DisfuncOrg', 'SIRSCritTachypnea', 'Hypotensie', 'SIRSCritHeartRate', 'Infusion', 'DiagnosticArtAstrup', 'DiagnosticIC', 'DiagnosticSputum', 'DiagnosticLiquor', 'DiagnosticOther', 'SIRSCriteria2OrMore', 'DiagnosticXthorax', 'SIRSCritTemperature', 'DiagnosticUrinaryCulture', 'SIRSCritLeucos', 'Oligurie', 'DiagnosticLacticAcid', 'lifecycle:transition', 'Diagnose', 'Hypoxie', 'DiagnosticUrinarySediment', 'DiagnosticECG']
  num_eventfts : ['Age', 'Leucocytes', 'CRP', 'LacticAcid']
Auto-derived window_size (98.5th percentile): 41
Auto-derived max_days (maximum case duration): 422.32
Generating Dataframes...


100%|███████████████████████████████████████████████| 26/26 [00:00<00:00, 564.98it/s]


Auto-detected features:
  cat_casefts  : ['case:REG_DATE', 'case:AMOUNT_REQ']
  num_casefts  : []
  cat_eventfts : ['org:resource', 'lifecycle:transition']
  num_eventfts : []
Auto-derived window_size (98.5th percentile): 17
Auto-derived max_days (maximum case duration): 89.51
Generating Dataframes...


100%|██████████████████████████████████████████████████| 5/5 [00:00<00:00, 16.91it/s]


Auto-detected features:
  cat_casefts  : ['case:REG_DATE', 'case:AMOUNT_REQ']
  num_casefts  : []
  cat_eventfts : ['org:resource', 'lifecycle:transition']
  num_eventfts : []
Auto-derived window_size (98.5th percentile): 8
Auto-detected features:
  cat_casefts  : ['case:Includes_subCases', 'case:Responsible_actor', 'case:endDate', 'case:caseStatus', 'case:parts', 'case:caseProcedure', 'case:last_phase', 'case:case_type', 'case:startDate', 'case:requestComplete', 'case:IDofConceptCase', 'case:termName', 'case:landRegisterID']
  num_casefts  : ['case:SUMleges']
  cat_eventfts : ['monitoringResource', 'org:resource', 'activityNameNL', 'question', 'dateFinished', 'action_code', 'activityNameEN', 'planned', 'lifecycle:transition', 'dueDate', 'dateStop']
  num_eventfts : []
Auto-derived window_size (98.5th percentile): 101
Auto-derived max_days (maximum case duration): 1326.00
Generating Dataframes...


...

Concurrent-trace test samples: 1791 / 1791
____________________________
Computing validation set tensors
  BPIC15_5  train=693 (21201 pairs)  val=174 (1176 pairs)  test=271 (1791 pairs)  conc=1791
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
____________________________
Computing test set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
Pairs – train: 22888  val: 2819  test: 1992
Tensors saved to 'results_per_log/BPIC15_1/'
tss_index = 1  (= 1 num_casefts + 0 num_eventfts)
Use this value for the tss_index parameter in TRAIN_EVAL_*.py scripts.


/workspace/baselines/create_general_data.py:553: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  _nb = torch.load(os.path.join(output_directory, 'test_new_block_labels.pt'))


Concurrent-trace test samples: 1992 / 1992
New-block labels saved to 'results_per_log/BPIC15_3/'.
Cases – train: 849  val: 213  test: 327
Generating Tensors...
Computing train set tensors
Generating prefix tensors ...
  BPIC15_1  train=709 (22888 pairs)  val=178 (2819 pairs)  test=295 (1992 pairs)  conc=1992
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
____________________________
Computing validation set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
____________________________
Computing test set tensors
Generating prefix tensors ...
Generating (decoder) suffix tensors ...
Generating time label tensors ...
Generating activity label tensors ...
Pairs – train: 26976  val: 2648  test: 2440
Tensors saved to 'results_per_log/BPIC15_3/'
tss_index = 1  (= 1 num_casefts + 0 num_eventfts)
Use this value for the tss_index para

/workspace/baselines/create_general_data.py:553: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  _nb = torch.load(os.path.join(output_directory, 'test_new_block_labels.pt'))


Concurrent-trace test samples: 2440 / 2440
  BPIC15_3  train=849 (26976 pairs)  val=213 (2648 pairs)  test=327 (2440 pairs)  conc=2440

Summary written to 'split_summary_sutran_best.csv'.


# Preproses all datasets Aston

In [2]:
from create_aston_data import run_all_logs

folder = "Logs/"
output_file = "split_summary_aston.csv"
run_all_logs(folder, output_file)

Processing 8 logs with 8 workers ...
[BPI_Challenge_2012_O] Re-running SuTraN dataframe pipeline ...
[BPI_Challenge_2012_A] Re-running SuTraN dataframe pipeline ...
[Sepsis] Re-running SuTraN dataframe pipeline ...
[BPIC15_2] Re-running SuTraN dataframe pipeline ...
[BPIC15_4] Re-running SuTraN dataframe pipeline ...


parsing log, completed traces ::   0%|          | 0/5015 [00:00<?, ?it/s]

[BPIC15_5] Re-running SuTraN dataframe pipeline ...
[BPIC15_1] Re-running SuTraN dataframe pipeline ...
[BPIC15_3] Re-running SuTraN dataframe pipeline ...

parsing log, completed traces ::   0%|          | 0/1050 [00:00<?, ?it/s]

parsing log, completed traces ::   0%|          | 0/13087 [00:00<?, ?it/s]

...

[BPIC15_1] Saved to '/workspace/baselines/results_per_log/BPIC15_1' (1082 distinct traces; instances train=22888, val=2819, test=1992)


[BPIC15_2] val replay:  63%|█████████████▎       | 1018/1610 [00:51<00:59,  9.98it/s]

  BPIC15_1  train=22888  val=2819  test=1992 pairs  traces=1082  net=380p/717t


[BPIC15_2] test replay: 100%|████████████████████| 1585/1585 [02:05<00:00, 12.63it/s]


[BPIC15_2] Saved to '/workspace/baselines/results_per_log/BPIC15_2' (743 distinct traces; instances train=17015, val=1610, test=1585)
  BPIC15_2  train=17015  val=1610  test=1585 pairs  traces=743  net=534p/843t

Summary written to 'split_summary_aston.csv'.


---
# Part 3 — Results Summary

After all cells above have finished, each model's results are stored as pickle files under:

```
<log_name>/
  SUTRAN_DA_results/TEST_SET_RESULTS/averaged_results.pkl
  SUTRAN_NDA_results/TEST_SET_RESULTS/averaged_results.pkl
  CRTP_LSTM_DA_results/TEST_SET_RESULTS/averaged_results.pkl
  CRTP_LSTM_ND_results/TEST_SET_RESULTS/averaged_results.pkl
  ED_LSTM_results/TEST_SET_RESULTS/averaged_results.pkl
  SEP_LSTM_results/TEST_SET_RESULTS/averaged_results.pkl
  BEST_results/TEST_SET_RESULTS/averaged_results.pkl
```

Each `averaged_results.pkl` is a dict with keys `"DL sim"`, `"MAE TTNE minutes"`, `"MAE RRT minutes"`.

The cell below collects and displays them in a summary table.

In [2]:
from SuffixTransformerNetwork.results_collector import get_suffix_baseline_results
import pandas as pd

EVENT_LOGS = [
    "BPIC15_1",
    "BPIC15_2",
    "BPIC15_3",
    "BPIC15_4",
    "BPIC15_5",
    #"RequestForPayment",
    "Sepsis",
    #"DomesticDeclarations",
    #"PrepaidTravelCost",
    #"InternationalDeclarations",
    #"BPI_Challenge_2012", # It have been splitted in 3 files A,O,W. Many papers do this.
    "BPI_Challenge_2012_A",
    "BPI_Challenge_2012_O",
    #"BPI_Challenge_2012_W",
    #"Hospital_Billing",
    #"Road_Traffic_Fine_Management_Process",
    #"BPI Challenge 2017"
]

df_results = get_suffix_baseline_results(EVENT_LOGS, base_dir="results_per_log")
for log_name, group in df_results.groupby(level="Log"):
    print(f"\n=== Log: {log_name} ===")

    print(
        group.reset_index()[["Model", 'Runs', 'DL similarity ↑ mean', 'DL similarity ↑ std',
       #'MAE TTNE (min) ↓ mean', 'MAE TTNE (min) ↓ std', 'MAE RRT (min) ↓ mean',
       #'MAE RRT (min) ↓ std'
        ]]
            
            
            
       #     [["Model", 'DL similarity ↑ mean', 'MAE TTNE (min) ↓ mean','MAE RRT (min) ↓ mean',
        #      'Conc N mean','Conc DL sim ↑ mean','Conc TTNE (min) ↓ mean', 
         #      'Conc RRT (min) ↓ mean', 'Conc next acc ↑ mean', 'Conc next F1 ↑ mean', '# params' ]]
        #[['Runs', 'DL similarity ↑ mean', 'DL similarity ↑ std',
       #'MAE TTNE (min) ↓ mean', 'MAE TTNE (min) ↓ std', 'MAE RRT (min) ↓ mean',
       #'MAE RRT (min) ↓ std']]
             .sort_values(by="DL similarity ↑ mean", ascending=False)
    )


=== Log: BPIC15_1 ===
             Model  Runs  DL similarity ↑ mean  DL similarity ↑ std
0     SuTraN (NDA)     0                   NaN                  NaN
1  CRTP-LSTM (NDA)     0                   NaN                  NaN
2          ED-LSTM     0                   NaN                  NaN
3         SEP-LSTM     0                   NaN                  NaN
4             BEST     0                   NaN                  NaN
5            ASTON     0                   NaN                  NaN

=== Log: BPIC15_2 ===
             Model  Runs  DL similarity ↑ mean  DL similarity ↑ std
0     SuTraN (NDA)     0                   NaN                  NaN
1  CRTP-LSTM (NDA)     0                   NaN                  NaN
2          ED-LSTM     0                   NaN                  NaN
3         SEP-LSTM     0                   NaN                  NaN
4             BEST     0                   NaN                  NaN
5            ASTON     0                   NaN                  NaN

=

In [2]:
df_results.columns

Index(['Runs', 'DL similarity ↑ mean', 'DL similarity ↑ std',
       'MAE TTNE (min) ↓ mean', 'MAE TTNE (min) ↓ std', 'MAE RRT (min) ↓ mean',
       'MAE RRT (min) ↓ std', 'Next act acc ↑ mean', 'Next act acc ↑ std',
       'Next act F1 (wt) ↑ mean', 'Next act F1 (wt) ↑ std', 'Conc N mean',
       'Conc N std', 'Conc DL sim ↑ mean', 'Conc DL sim ↑ std',
       'Conc TTNE (min) ↓ mean', 'Conc TTNE (min) ↓ std',
       'Conc RRT (min) ↓ mean', 'Conc RRT (min) ↓ std', 'Conc next acc ↑ mean',
       'Conc next acc ↑ std', 'Conc next F1 ↑ mean', 'Conc next F1 ↑ std',
       '# params', 'training_time', 'testing_time'],
      dtype='object')